<a href="https://colab.research.google.com/github/kerenG99/UFZ-Helmoltz-Summer-School-2026/blob/main/notebooks/05_capstone_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 · Capstone — a guided microbiome analysis in R

> **Trends in multi-omics data analysis for Microbial Ecology and Biotechnology** - Summer School, UFZ Leipzig  
> *Refreshing / Basic Knowledge: R* - Instructor: Anderson Santos

## Setup — run this first

This cell installs the R packages the lessons use and makes the example data available. On **Google Colab** it just works: run it (Shift+Enter) and wait until it prints **Ready**. You do not need to understand it.

In [ ]:
# Setup — run this first. Works on Google Colab and on a local Jupyter with R.
# It installs the packages the lessons use and makes the example data available.

pkgs <- c("readr", "dplyr", "tidyr", "tibble", "ggplot2", "forcats", "stringr", "vegan")
to_install <- setdiff(pkgs, rownames(installed.packages()))
if (length(to_install) > 0) {
  # On Colab (Linux) use Posit Public Package Manager binaries: ~1-2 min instead of
  # compiling from source (~15-20 min).
  if (grepl("linux", R.version$os) && file.exists("/etc/os-release")) {
    cn <- grep("^VERSION_CODENAME=", readLines("/etc/os-release"), value = TRUE)
    cn <- gsub('VERSION_CODENAME=|"', "", cn)
    if (length(cn) == 1 && nzchar(cn)) {
      options(repos = c(CRAN = sprintf("https://p3m.dev/cran/__linux__/%s/latest", cn)))
      options(HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
              paste(getRversion(), R.version$platform, R.version$arch, R.version$os)))
    }
  }
  install.packages(to_install)
}

# Fetch the example data if it is not already next to the notebook (the Colab case).
if (!file.exists("../data/sample_metadata.csv") && !file.exists("data/sample_metadata.csv")) {
  system("git clone -q https://github.com/andersonavilasantos/ufz-summerschool-r.git course_repo")
  setwd("course_repo/notebooks")
}

# Load the packages quietly and keep read_csv output tidy (cleaner notebook output).
suppressPackageStartupMessages(invisible(lapply(pkgs, require, character.only = TRUE)))
options(readr.show_col_types = FALSE)

cat("Ready. Data folder:", if (file.exists("../data/sample_metadata.csv")) "../data" else "data", "\n")

In [ ]:
#github comit

## The scenario

You have received a human-gut microbiome study — three tables: **abundances**
(1,006 samples × 130 genera), a **taxonomy**, and **participant metadata**. Your
job: take it from raw tables to a few defensible conclusions, combining
everything from lessons 01–04.

## How this notebook works

It is a **hands-on activity**, not a lecture. Each **step** states a task and
gives hints; you write the code in the blank cell. A **collapsible solution**
sits under each step — try first, peek if stuck. Steps build on each other, so
run them in order.

> Suggested time: 45–60 min. Steps 1–7 are core; 8–10 are extension.

> **Instructor note.** Run as pair-programming: one drives, one navigates.
> Reveal each solution only after most pairs have attempted the step.

---

## Step 0 · Setup — load packages and data (just run it)

In [ ]:
library(readr); library(dplyr); library(tidyr)          # import + wrangle
library(ggplot2); library(forcats); library(stringr)    # plot + factors + text
library(vegan)     # community ecology toolkit: diversity() and ordination (metaMDS)

# Locate and load the three tables (same pattern used in every lesson).
data_dir <- if (file.exists("../data/sample_metadata.csv")) "../data" else "data"
meta     <- read_csv(file.path(data_dir, "sample_metadata.csv"))
taxonomy <- read_csv(file.path(data_dir, "taxonomy.csv"))
abund    <- read_csv(file.path(data_dir, "genus_abundance.csv"))

# cat() prints a quick shape report so we can confirm the loads worked.
cat("metadata :", nrow(meta), "x", ncol(meta), "\n")
cat("taxonomy :", nrow(taxonomy), "x", ncol(taxonomy), "\n")
cat("abundance:", nrow(abund), "x", ncol(abund), "\n")

---

## Step 1 · Clean the metadata

**Task.** Real data is messy. (a) Report how many rows have a missing `age` and
how many have a missing `sex`. (b) Make an analysis-ready copy `meta_clean` that
keeps only rows with a known `age` **and** known `bmi_group`, and set
`bmi_group` as an **ordered factor** (underweight → morbidly obese).

*Hints:* `is.na()`, `sum()`, `filter()`, `factor(..., levels = ...)`.

In [ ]:
# Your code here

<details><summary><b>Solution: Step 1</b></summary>


```r
# is.na() flags missing values; sum() counts the TRUEs -> how many gaps in each column.
cat("missing age:", sum(is.na(meta$age)),
    " | missing sex:", sum(is.na(meta$sex)), "\n")

bmi_levels <- c("underweight","lean","overweight","obese","severeobese","morbidobese")
meta_clean <- meta |>
  filter(!is.na(age), !is.na(bmi_group)) |>              # keep only fully-labelled samples
  mutate(bmi_group = factor(bmi_group, levels = bmi_levels))  # + set the clinical order

cat("kept", nrow(meta_clean), "of", nrow(meta), "samples\n")  # show the cost of missing data
```
</details>

---

## Step 2 · Reshape abundances to tidy long form

**Task.** Turn the wide `abund` table (one column per genus) into a **long** table
`ab_long` with columns `sample_id`, `genus`, `abundance`. Report its number of
rows and confirm it equals samples × genera.

*Hints:* `pivot_longer(-sample_id, names_to = ..., values_to = ...)`.

In [ ]:
# Your code here

<details><summary><b>Solution: Step 2</b></summary>


```r
# pivot_longer stacks the 130 genus columns into two: 'genus' (name) + 'abundance' (value).
ab_long <- abund |>
  pivot_longer(-sample_id, names_to = "genus", values_to = "abundance")
# Sanity check: rows should equal samples x genera (ncol - 1 drops the sample_id column).
cat(nrow(ab_long), "rows =", nrow(abund), "samples x", ncol(abund) - 1, "genera\n")
```
</details>

---

## Step 3 · Join taxonomy and metadata

**Task.** Build one tidy table `ab_full` = `ab_long` + phylum/family (from
`taxonomy`, by `genus`) + region/BMI/age (from `meta_clean`, by `sample_id`).
Because `meta_clean` dropped some samples, keep only rows that have metadata
(`inner_join`, or drop `NA` region).

*Hints:* `left_join()` / `inner_join()` on the shared keys.

In [ ]:
# Your code here

<details><summary><b>Solution: Step 3</b></summary>


```r
ab_full <- ab_long |>
  left_join(taxonomy, by = "genus") |>                    # add phylum/family per genus
  # inner_join keeps only rows that match on both sides -> drops samples cleaned away.
  inner_join(select(meta_clean, sample_id, nationality, bmi_group, age),
             by = "sample_id")
glimpse(ab_full)
```
</details>

---

## Step 4 · Alpha diversity per sample (compute it yourself)

**Task.** For every sample compute (a) **observed richness** (number of genera
with abundance > 0) and (b) the **Shannon index** on the abundances. Put them in
`alpha`, then join `meta_clean` and report the **mean of each metric per region**.

*Hints:* group `ab_full` by `sample_id`; richness = `sum(abundance > 0)`;
Shannon = `diversity(abundance, index = "shannon")` from **vegan** (feed it the
per-sample abundance vector), or compute by hand.

In [ ]:
# Your code here

<details><summary><b>Solution: Step 4</b></summary>


```r
# Group by sample (carrying its metadata) so summarise() gives one diversity row per sample.
alpha <- ab_full |>
  group_by(sample_id, nationality, bmi_group, age) |>
  summarise(
    richness = sum(abundance > 0),                       # count genera present (abundance > 0)
    shannon  = diversity(abundance, index = "shannon"),  # vegan's Shannon index on the vector
    .groups  = "drop")

# Then average each metric per region to compare communities.
alpha |>
  group_by(nationality) |>
  summarise(mean_richness = mean(richness),
            mean_shannon  = mean(shannon),
            n = n()) |>                                  # n() = samples per region
  arrange(desc(mean_shannon))                            # most diverse region first
```
</details>

---

## Step 5 · A diversity figure

**Task.** Make a **boxplot** of `shannon` across `bmi_group` (from `alpha`),
filled by BMI group, with clear labels and no redundant legend. Bonus: overlay
faint jittered points.

*Hints:* `geom_boxplot()`, `geom_jitter(alpha = ...)`, `labs()`,
`theme(legend.position = "none")`.

In [ ]:
# Your code here

<details><summary><b>Solution: Step 5</b></summary>


```r
ggplot(alpha, aes(bmi_group, shannon, fill = bmi_group)) +  # category (x) vs number (y)
  geom_boxplot(outlier.alpha = 0.3) +                       # median + quartiles per BMI group
  geom_jitter(width = 0.15, alpha = 0.1, size = 0.4) +      # faint raw points show sample sizes
  labs(title = "Gut diversity across BMI groups",
       x = "BMI group", y = "Shannon diversity") +
  theme_minimal() + theme(legend.position = "none")         # x already labels groups -> no legend
```
</details>

---

## Step 6 · Taxonomic composition per region

**Task.** Compute the **mean relative abundance of each phylum per region** and
draw a **stacked bar chart** (y as %). Name the dominant phylum in each region.

*Hints:* per sample → per-phylum sum → divide by the sample total → average per
region; `geom_col()`, `scale_y_continuous(labels = scales::percent)`.

In [ ]:
# Your code here

<details><summary><b>Solution: Step 6</b></summary>


```r
# Three-stage split-apply-combine: per-phylum sum -> per-sample share -> region mean.
composition <- ab_full |>
  group_by(sample_id, nationality, phylum) |>
  summarise(phy = sum(abundance), .groups = "drop") |>  # total abundance per phylum in a sample
  group_by(sample_id) |>
  mutate(rel = phy / sum(phy)) |>                       # divide by sample total -> proportion
  group_by(nationality, phylum) |>
  summarise(mean_rel = mean(rel), .groups = "drop")     # average that share per region

ggplot(composition, aes(nationality, mean_rel,
                        fill = fct_reorder(phylum, mean_rel, sum))) +  # order the stack by size
  geom_col() +                                          # stacks automatically (fill varies)
  scale_y_continuous(labels = scales::percent) +        # y axis as %
  labs(title = "Phylum composition across regions",
       x = NULL, y = "Mean relative abundance", fill = "Phylum") +
  theme(axis.text.x = element_text(angle = 30, hjust = 1))  # tilt long region labels

# slice_max keeps the single top-ranked phylum within each region group:
composition |> group_by(nationality) |> slice_max(mean_rel, n = 1)
```
</details>

---

## Step 7 · A simple statistic — does diversity differ by BMI?

**Task.** Test whether Shannon diversity differs across BMI groups. Use a
**one-way ANOVA** (`aov`) of `shannon ~ bmi_group` on `alpha`, and report the
p-value. State in one sentence whether there is evidence of a difference.

*Hints:* `fit <- aov(shannon ~ bmi_group, data = alpha)`; `summary(fit)`.

In [ ]:
# Your code here

<details><summary><b>Solution: Step 7</b></summary>


```r
fit <- aov(shannon ~ bmi_group, data = alpha)
summary(fit)
# The p-value (Pr(>F)) tells you whether mean diversity differs across BMI
# groups. Interpret it in plain words for the class; pair it with the Step 5 boxplot.
```
</details>

---

## Step 8 · Ordination — NMDS of community structure (extension)

**Task.** Do samples from different regions form different **communities**? Build
a sample × genus matrix, run **NMDS** on Bray–Curtis distances with **vegan**, and
plot the two ordination axes coloured by region.

*Hints:* `pivot_wider()` back to a matrix (drop `sample_id` to a rowname);
`metaMDS(mat, distance = "bray")`; extract `scores(..., display = "sites")`.

In [ ]:
# Your code here

<details><summary><b>Solution: Step 8</b></summary>


```r
# set.seed() fixes the random draw so everyone in class gets the same subset/ordination.
set.seed(1)
# Take a balanced subset: 40 random samples from each of four regions (fast enough live).
sub_ids <- meta_clean |>
  filter(nationality %in% c("Scandinavia","US","SouthEurope","CentralEurope")) |>
  group_by(nationality) |> slice_sample(n = 40) |> pull(sample_id)  # pull() -> a plain vector

# vegan wants a numeric matrix with samples as rows and genera as columns.
mat <- abund |>
  filter(sample_id %in% sub_ids) |>
  tibble::column_to_rownames("sample_id") |>  # move sample_id from a column to row names
  as.matrix()
mat <- mat[rowSums(mat) > 0, ]              # keep only rows (samples) with any reads

# metaMDS: NMDS on Bray-Curtis distances; try = 20 restarts to find a stable solution.
nmds <- metaMDS(mat, distance = "bray", trace = FALSE, try = 20)
```


```r
# scores(..., "sites") = the 2-D coordinates of each sample; turn them into a data frame.
scores_df <- as.data.frame(scores(nmds, display = "sites"))
scores_df$sample_id <- rownames(scores_df)  # recover sample_id from the row names
scores_df <- left_join(scores_df,           # attach each sample's region for colouring
                       select(meta_clean, sample_id, nationality), by = "sample_id")

ggplot(scores_df, aes(NMDS1, NMDS2, colour = nationality)) +  # the two ordination axes
  geom_point(size = 2, alpha = 0.8) +
  stat_ellipse(level = 0.7) +               # a 70% confidence ellipse summarising each region
  labs(title = paste0("NMDS of gut communities (stress = ",
                      round(nmds$stress, 3), ")"),  # stress in the title (lower = better 2-D fit)
       colour = "Region") +
  theme_minimal()
```
</details>

---

## Step 9 · Export your results (extension)

**Task.** Save two outputs: (a) the `alpha` diversity table and (b) the
`composition` table, as CSV files, so a collaborator can use them.

*Hints:* `write_csv(x, "path.csv")`.

In [ ]:
# Your code here

<details><summary><b>Solution: Step 9</b></summary>


```r
# write_csv() is the inverse of read_csv(): save a tibble back to a shareable CSV file.
write_csv(alpha,       "capstone_alpha_diversity.csv")   # per-sample diversity metrics
write_csv(composition, "capstone_phylum_by_region.csv")  # phylum composition per region
cat("Wrote capstone_alpha_diversity.csv and capstone_phylum_by_region.csv\n")
```
</details>

---

## Step 10 · Conclude

Write **2–3 sentences** answering: *which regions or BMI groups differ most in
their gut communities, and which taxa drive the difference?* Base it on your
figures and the ANOVA. Then note **one thing you would check next** (e.g. adjust
for age, rarefy to equal depth, test region with PERMANOVA `vegan::adonis2`).

> *(Double-click this text cell to edit it and type your interpretation.)*
>
> - Groups that differ most: …
> - Taxa driving the difference: …
> - One thing I would check next: …

---

## Wrap-up

You have now run a complete microbial-ecology workflow in R: import, clean,
reshape, join, summarise, visualise, test, ordinate, and export. Every tool came
from lessons 01–04; the only new thing was chaining them into an analysis.

Keep this notebook as a template. Swap in your own feature table this week and
most of the code still applies.

To see the whole thing written out, with every cell filled in and all figures
produced, open **07 · Worked analysis**, the fully worked, read-and-run version of
this capstone.